# B01 — Batch Ingestion and Delta Engineering

**Time: about 75 minutes.**
Covers: Batch Processing Fundamentals · Ingesting Batch Data · Data Engineering with Delta Lake

### What you will be able to do afterwards

- Ingest files idempotently, so re-running a load is safe rather than destructive.
- Capture file-level metadata at ingestion and explain why you would want it.
- Absorb a new source column without failing the load or losing the values.
- Apply a CDC feed — inserts, updates and deletes — with `MERGE INTO`.
- Implement a soft delete and justify it against a hard delete.

### The scenario

The sales system sends one file per day. Day 0 is a full snapshot; every day after is a
delta where each row carries a `status` of `insert`, `update` or `delete`. On day 4 the
source team adds a `region` column without telling anybody.

Your job is to build a pipeline that can be re-run on any day without corrupting anything.

In [ ]:
from pyspark.sql import DataFrame, functions as F
from helpers import utils

cfg = utils.get_configs("sales")
catalog, bronze_schema, silver_schema = cfg["catalog"], cfg["schema_bronze"], cfg["schema_silver"]
bronze_table, silver_table = cfg["table_bronze"], cfg["table_silver"]
raw_path, checkpoint_path, schema_path = cfg["raw_path"], cfg["checkpoint_path"], cfg["schema_path"]

spark, dbutils = utils.spark, utils.dbutils

print(f"bronze     {bronze_table}")
print(f"silver     {silver_table}")
print(f"raw        {raw_path}")
print(f"checkpoint {checkpoint_path}")
print(f"schema     {schema_path}")

## Step 0 — Look at what you were sent

Before writing any ingestion code, read the files stored in raw schema. The path is already in your config cell from earlier in the notebook: raw_path

**TO DO**

1. List the sales files. Note how many there are and how they are named.
2. Print the header of the **first** file and of the **last** file. They differ. You can use `dbutils.fs`/`spark.read` directly against the CSVs sitting in the volume
3. Read one file into a DataFrame and look at the distinct values of `status`.

> **Question:** the day-0 file and the day-5 file have different columns. Name two ways a
> pipeline could handle that, and what each one costs you.

In [ ]:
# TO DO: list the sales files


# TO DO: print the head of the first and last file — compare the headers


# TO DO: read one delta file and inspect the status column

## Step 1 — Ingest into bronze, idempotently

**TO DO**

Load every sales CSV into a bronze Delta table. Use **either** `COPY INTO` **or** Auto
Loader with `trigger(availableNow=True)` — both track which files they have already
consumed, which is the property that matters here.

Requirements:

- Capture the source file name into a column called `source_file`. Both `COPY INTO` and
  Auto Loader expose the `_metadata` struct: `_metadata.file_name`,
  `_metadata.file_modification_time`, `_metadata.file_size`.
- Keep `status` and `updated_at`. Bronze records what you were sent, unedited.
- Do not deduplicate here.

**Tips**

- `COPY INTO` needs the target table to exist first. Auto Loader can create it via `.toTable()`.
- With Auto Loader, use `checkpoint_path` and `schema_path` from the config cell.

> **Question:** "idempotent" here means re-running skips files already consumed. What
> exactly is doing the remembering in the option you picked, and where does it live?

In [ ]:
def create_bronze_sales(full_table_name: str) -> None:
    """Create the bronze sales table. Optional if you use Auto Loader with .toTable()."""
    # TO DO
    pass


def ingest_sales(full_table_name: str, data_path: str) -> None:
    """
    Load every sales CSV into bronze, recording the source file name.

    Args:
        full_table_name: 'catalog.schema.table'.
        data_path: volume directory holding the CSVs.
    """
    # TO DO
    pass


# TO DO: run the ingestion, then count the rows

## Step 2 — Prove it is idempotent

**TO DO**

1. Record the current row count. You can use `utils.get_table_row_count(table_name)` function.
2. Run your ingestion function **again**, unchanged.
3. Compare the row count. It must be identical.
4. Look at `DESCRIBE HISTORY` — was a new version committed even though no rows changed?

> **Question:** now delete the checkpoint directory (or use a fresh `COPY INTO` target) and
> re-run. What happens, and what does that tell you about where the idempotency actually
> comes from? Do not leave the table in that state — the later checks assume one copy of
> each file.

In [ ]:
# TO DO: count, re-run, count again, compare


# TO DO: inspect the history

## Step 3 — Absorb the new column

The later sales files carry a `region` column. A naive load either fails or silently drops
the values.

**TO DO**

1. Confirm `region` is now in your bronze table and that some rows have a value.
2. Confirm the early rows have `region` as null rather than having been rejected.
3. If your Step 1 load dropped it, fix the load and re-run.

**Tips**

- `COPY INTO`: `COPY_OPTIONS ('mergeSchema' = 'true')` on the write side, and
  `FORMAT_OPTIONS ('mergeSchema' = 'true')` on the read side. You need both.
- Auto Loader: `cloudFiles.schemaEvolutionMode = 'addNewColumns'`, plus `mergeSchema` on
  the write. The stream stops the first time it sees the new column — that is by design;
  restart it and it picks up from where it left off.

> **Question:** `addNewColumns` adds columns but never widens an existing column's type.
> If `price` arrived as `"19.99 USD"` tomorrow, what happens, and which column would you
> look in to find out?

In [ ]:
# TO DO: confirm region exists and is populated for later rows only


# TO DO: check the schema

## Step 4 — Build the silver CDC table

Bronze is the raw feed. Silver is the current state of every sale.

**TO DO**

Create the silver table with this schema:

| Column | Type | Notes |
| :-- | :-- | :-- |
| `sale_id` | `INT` | merge key |
| `product_id` | `INT` | |
| `user_id` | `INT` | |
| `quantity` | `INT` | renamed from `qty` |
| `price` | `DOUBLE` | |
| `region` | `STRING` | |
| `_is_active` | `BOOLEAN` | false once the source deletes the sale |
| `_created_at` | `DATE` | `updated_at` of the row that first inserted it |
| `_updated_at` | `DATE` | `updated_at` of the row that last changed it |
| `_source_file` | `STRING` | which delivery last touched this row |

`status` and `updated_at` drive the merge and then do not survive into silver — they are
instructions, not data.

In [ ]:
def create_silver_sales(full_table_name: str) -> None:
    """Create the empty silver sales table."""
    # TO DO
    pass


# TO DO: create it and confirm the schema

## Step 5 — Apply the CDC feed, one day at a time

**TO DO**

Write `cdc_merge(bronze, silver, filter_date)` that applies exactly one day of changes:

1. Filter bronze to rows where `updated_at` equals `filter_date`.
2. If a `sale_id` appears more than once within that day, keep only the last change.
3. `MERGE INTO` silver on `sale_id`:
   - **matched** and `status = 'delete'` → set `_is_active = false`, update `_updated_at`
   - **matched** otherwise → update the business columns and `_updated_at`, leave `_created_at` alone
   - **not matched** and `status <> 'delete'` → insert with `_is_active = true` and both
     timestamps set to `updated_at`
   - **not matched** and `status = 'delete'` → do nothing. A delete for a sale you never
     saw is a no-op, not an error.

Then call it once per date, in order.

**Why one day at a time.** You could merge the whole feed in a single statement. Applying
it per day means that when day 4 fails at 3am you re-run day 4, not the entire history —
and you can prove the pipeline is replayable by re-running any single day.

> **Questions:**
> - Why keep `_created_at` unchanged on update? What breaks in the gold layer if you don't?
> - Re-run one day that you have already applied. Does anything change? Should it?

In [ ]:
def cdc_merge(bronze: str, silver: str, filter_date: str) -> None:
    """
    Apply one day of CDC changes from bronze into silver.

    Args:
        bronze: fully qualified bronze table.
        silver: fully qualified silver table.
        filter_date: 'YYYY-MM-DD'.
    """
    # TO DO
    pass


def all_change_dates(bronze: str) -> list:
    """Distinct updated_at values in bronze, oldest first."""
    # TO DO
    pass


# TO DO: loop over the dates in order and merge each one
for filter_date in all_change_dates(bronze_table):
    pass

## Step 6 — Validate

**TO DO**

1. Count active versus soft-deleted rows in silver.
2. Confirm no `sale_id` appears twice.
3. `DESCRIBE HISTORY` on silver — you should see one `MERGE` per day.

In [ ]:
# TO DO: active vs inactive counts, duplicate check, history


# TO DO: trace one sale_id through its whole lifecycle

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("B01-batch-ingestion-and-delta")

## Recap

- Idempotency comes from something remembering which files were consumed — a checkpoint or
  `COPY INTO`'s own tracking. It is not a property of the SQL you wrote.
- `_metadata` costs nothing at ingestion and is the first column you will want during an
  incident.
- Schema evolution needs handling on both the read and the write side. Getting one right
  and not the other is the usual cause of a null column.
- CDC applied per day is replayable per day. That is the difference between a five-minute
  fix and a full reload.